##### Install libraries


In [ ]:
%pip install datasets

In [ ]:
%pip install transformers

In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

##### Import & cut dataset

In [ ]:

from datasets import load_dataset, Dataset
from transformers import AutoTokenizer
import random

# Parameters
SEED = 123
DATA_PERCANTAGE = 0.02
TRAIN_RATIO = 0.9
VAL_RATIO = (1 - TRAIN_RATIO) * 0.5
TEST_RATIO = (1 - TRAIN_RATIO) * 0.5

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Load english wikipedia streaming dataset
stream = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True
)

# Select a percentage of articles
texts = []
for example in stream:
    if random.random() < DATA_PERCANTAGE:
        texts.append(example["text"])

print(f"Selected articles: {len(texts)}")

dataset = Dataset.from_dict({"text": texts})

# Shuffle and split the dataset
random.seed(SEED)
dataset = dataset.shuffle(seed=SEED)

train_test = dataset.train_test_split(test_size=(1 - TRAIN_RATIO), seed=SEED)

test_val = train_test["test"].train_test_split(
    test_size=TEST_RATIO / (TEST_RATIO + VAL_RATIO),
    seed=SEED
)

train_ds = train_test["train"]
val_ds   = test_val["train"]
test_ds  = test_val["test"]

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

# Count tokens
def count_tokens(ds):
    total = 0
    for t in ds["text"]:
        total += len(tokenizer(t)["input_ids"])
    return total

train_tokens = count_tokens(train_ds)
val_tokens   = count_tokens(val_ds)
test_tokens  = count_tokens(test_ds)

print("\nToken counts:")
print(f"Train: {train_tokens:,}")
print(f"Val:   {val_tokens:,}")
print(f"Test:  {test_tokens:,}")
print(f"Total: {train_tokens + val_tokens + test_tokens:,}")

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Selected documents: 128225


Token indices sequence length is longer than the specified maximum sequence length for this model (1391 > 1024). Running this sequence through the model will result in indexing errors


Train: 115402  Val: 6411  Test: 6412

Token counts:
Train: 83,123,774
Val:   4,513,580
Test:  4,800,950
Total: 92,438,304


##### Preview dataset

In [ ]:
def preview(ds, name):
    print(f"\n===== {name} PREVIEW =====")
    for i in range(min(3, len(ds))):
        text = ds[i]["text"]
        tokens = tokenizer(text)["input_ids"]

        print(f"\n--- Sample {i+1} ---")
        print("Text (first 300 chars):")
        print(text[:300].replace("\n", " ") + " ...")

        print("\nFirst 20 token ids:")
        print(tokens[:20])

        print("\nDecoded tokens:")
        print(tokenizer.convert_ids_to_tokens(tokens[:20]))

preview(train_ds, "TRAIN")


===== TRAIN PREVIEW =====

--- Sample 1 ---
Text (first 300 chars):
The Nguyen Thien Thuat apartment buildings (Vietnamese: Chung cư Nguyễn Thiện Thuật) are a complex of American-built historic buildings in District 3, Ho Chi Minh City. The apartments are located on Nguyen Thien Thuat street, a thoroughfare known for its musical instrument shops.  Constructed in 196 ...

First 20 token ids:
[464, 42379, 536, 2013, 26223, 265, 7962, 6832, 357, 53, 1155, 22678, 25, 43915, 269, 130, 108, 399, 22932, 157]

Decoded tokens:
['The', 'ĠNguyen', 'ĠTh', 'ien', 'ĠThu', 'at', 'Ġapartment', 'Ġbuildings', 'Ġ(', 'V', 'iet', 'namese', ':', 'ĠChung', 'Ġc', 'Æ', '°', 'ĠN', 'guy', 'á']

--- Sample 2 ---
Text (first 300 chars):
Blackbird is a 2019 American drama film directed by Roger Michell and written by Christian Torpe. It is a remake of the 2014 Danish film Silent Heart, also written by Torpe. It stars Susan Sarandon, Kate Winslet, Mia Wasikowska, Lindsay Duncan, Rainn Wilson, Bex Taylor-Klaus, and 